In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "stsb"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1379
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 5749
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1379
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'A plane is taking off.',
 'sentence2': 'An air plane is taking off.',
 'label': 5.0,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1']

'Some men are fighting.'

In [11]:
raw_train_dataset[5]['sentence2']

'Two men are fighting.'

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': Value('float32'),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 1500/1500 [00:00<00:00, 38094.25 examples/s]


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1379
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 66]),
 'token_type_ids': torch.Size([32, 66]),
 'attention_mask': torch.Size([32, 66])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
num_labels = 1

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    problem_type="regression",
    # id2label=id2label,
    # label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
teacher_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    problem_type="regression",
    # id2label=id2label,
    # label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
teacher_model.set_use_module_grafting(False)
teacher_model.set_use_scc_status(False)

### Load Teacher Model

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location='cpu')
teacher_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
teacher_model.to(device)

device

device(type='cuda')

In [28]:
from train_eval_func import eval_loop

In [29]:
eval_loop(teacher_model, val_dataloader, task_name, device, regression=True)

({'pearson': 0.900084701022039, 'spearmanr': 0.8962584593578651},
 0.4520208261748578)

# 5. Layer Score Importance And Sorting

In [30]:
from structure_prune_helper import compute_layer_importance_scores, register_importance_masks, remove_importance_masks

In [31]:
register_importance_masks(teacher_model, device, register_mha_layer=True, register_mlp_layer=True,
                         register_mha=True, register_mlp=True)

  Layer 0: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 1: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 2: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 3: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 4: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 5: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 6: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 7: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 8: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 9: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 10: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 11: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [32]:
from layer_merge_helper import reorder_merged_layer_heads

In [33]:
mha_imp, ffn_imp, head_imp, neuron_imp, sublayer_imp = compute_layer_importance_scores(
    teacher_model, train_dataloader, device,
    compute_mha=True, compute_ffn=True,
    compute_heads=True, compute_neurons=True,
    max_batches=115
)

Computing importance (MHA layer, FFN layer, heads, neurons): 100%|██████████| 115/115 [00:10<00:00, 11.06it/s]


Layer Importance Summary (separate):
          Layer     MHA(layer)     FFN(layer)    Heads(mean)  Neurons(mean)
  ---------------------------------------------------------------------------
  0                     0.8654         0.7192         0.0000         0.0000
  1                     0.5033         0.6466         0.0000         0.0000
  2                     1.0000         0.5960         0.0000         0.0000
  3                     0.5320         0.5350         0.0000         0.0000
  4                     0.6892         1.0000         0.0000         0.0000
  5                     0.9257         0.7986         0.0000         0.0000
  6                     0.5441         0.5591         0.0000         0.0000
  7                     0.4191         0.4553         0.0000         0.0000
  8                     0.4512         0.5134         0.0000         0.0000
  9                     0.4123         0.3721         0.0000         0.0000
  10                    0.4720         0.3829   

In [34]:
for i in range(12):
    reorder_merged_layer_heads(model.bert.encoder.layer[i], head_imp[i].cpu(), torch.device("cpu"))

Reordered 12 heads, new order: [8, 0, 5, 10, 1, 9, 3, 11, 4, 7, 2, 6]
Reordered 12 heads, new order: [11, 7, 8, 1, 6, 2, 4, 10, 9, 0, 3, 5]
Reordered 12 heads, new order: [6, 2, 10, 3, 0, 11, 9, 7, 5, 1, 4, 8]
Reordered 12 heads, new order: [0, 7, 10, 4, 3, 1, 5, 6, 2, 8, 11, 9]
Reordered 12 heads, new order: [3, 1, 2, 9, 4, 5, 0, 10, 7, 6, 11, 8]
Reordered 12 heads, new order: [8, 0, 5, 10, 4, 2, 7, 9, 11, 3, 1, 6]
Reordered 12 heads, new order: [0, 8, 4, 7, 5, 1, 9, 11, 10, 6, 2, 3]
Reordered 12 heads, new order: [10, 5, 0, 8, 4, 9, 2, 1, 11, 6, 3, 7]
Reordered 12 heads, new order: [8, 9, 1, 3, 7, 10, 2, 6, 11, 5, 4, 0]
Reordered 12 heads, new order: [8, 11, 10, 6, 5, 7, 4, 0, 9, 3, 2, 1]
Reordered 12 heads, new order: [3, 10, 0, 11, 8, 7, 9, 2, 5, 4, 6, 1]
Reordered 12 heads, new order: [9, 2, 4, 1, 3, 8, 10, 6, 0, 5, 7, 11]


In [35]:
layer_scores = []
for idx in range(0, 24, 2):
    layer = int(idx / 2)
    layer_score = (sublayer_imp[idx] + sublayer_imp[idx+1]) / 2
    print(layer, layer_score)
    layer_scores.append(layer_score)

0 0.7630165250651573
1 0.5579402261297681
2 0.7641968843151303
3 0.5155075368269233
4 0.8212861392644557
5 0.830862729665943
6 0.5332154394448694
7 0.42303025399555305
8 0.46706087212761127
9 0.37824012670181756
10 0.4114474005317562
11 0.6113214004578735


# 6. Calculating Similarity 

In [36]:
from CKA import CKAEvaluator

cka_eval = CKAEvaluator(device)

In [37]:
reps_similarity, atts_similarity, sub_similarity = cka_eval.subLayer_interleaved_pairwise(
    teacher_model, 
    train_dataloader, 
    device, 
    only_cls_token=True, 
    max_iter=115
)

CKA Sublayer Evaluation: 100%|██████████| 115/115 [00:29<00:00,  3.85it/s]


In [38]:
reps_stats = cka_eval.similarity_stats(reps_similarity)
atts_stats = cka_eval.similarity_stats(atts_similarity)
subs_stats = cka_eval.similarity_stats(sub_similarity)

print("  Reps Similarity Stats:")
print(f"    Average: {reps_stats['average']:.6f}")
print(f"    Highest: {reps_stats['max']:.6f}")
print(f"    Lowest:  {reps_stats['min']:.6f}")
print("")

print("  Atts Similarity Stats:")
print(f"    Average: {atts_stats['average']:.6f}")
print(f"    Highest: {atts_stats['max']:.6f}")
print(f"    Lowest:  {atts_stats['min']:.6f}")

print("  Sublayer Similarity Stats:")
print(f"    Average: {subs_stats['average']:.6f}")
print(f"    Highest: {subs_stats['max']:.6f}")
print(f"    Lowest:  {subs_stats['min']:.6f}")

  Reps Similarity Stats:
    Average: 0.924855
    Highest: 0.993679
    Lowest:  0.547793

  Atts Similarity Stats:
    Average: 0.935733
    Highest: 0.992232
    Lowest:  0.787959
  Sublayer Similarity Stats:
    Average: 0.364992
    Highest: 0.656474
    Lowest:  0.190959


# 7. Creating Student Model

### Layer Composition

In [39]:
layer_composition = [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]

In [40]:
from layer_merge_helper import merge_high_sim_layer_composition

In [41]:
layer_composition = merge_high_sim_layer_composition(layer_composition, sublayer_imp, reps_similarity, layer_scores, 0.9)

In [42]:
layer_composition, len(layer_composition)

([[0, 1], [2], [3], [4], [5, 6, 7, 8, 9, 10], [11]], 6)

In [43]:
print(layer_scores)

[0.7630165250651573, 0.7630165250651573, 0.7641968843151303, 0.5155075368269233, 0.8212861392644557, 0.830862729665943, 0.830862729665943, 0.830862729665943, 0.830862729665943, 0.830862729665943, 0.830862729665943, 0.6113214004578735]


### Merging Layers

In [44]:
from layer_merge_helper import merge_mha_cluster

In [45]:
import math

In [46]:
mha_mlp_contribution = {}

In [47]:
for layers in layer_composition:
    if len(layers) == 1:
        continue
    print(layers)

    head_total_imp = 0
    # Track highest MLP importance
    mlp_highest_score = -float("inf")
    mlp_highest_layer = None

    # ----------------------------
    # Calculate cluster statistics
    # ----------------------------
    for layer in layers:
        mha_score = sublayer_imp[layer * 2]
        mlp_score = sublayer_imp[layer * 2 + 1]
        
        # total MHA importance
        head_total_imp += mha_score
        
        # find strongest FFN layer
        if mlp_score > mlp_highest_score:
            mlp_highest_score = mlp_score
            mlp_highest_layer = layer
            
    print(head_total_imp)
    print("Layer with highest mlp score:", mlp_highest_layer)
    obj = {"n_heads_per_layer": {}, "mlp_highest_layer": mlp_highest_layer}

    for layer in layers:
        head_graft_n = math.ceil((12 * sublayer_imp[layer*2]) / head_total_imp)
        print(f'Layer: {layer}, Number of Heads: {head_graft_n}')
        obj['n_heads_per_layer'][layer] = head_graft_n

    mha_mlp_contribution[layer] = obj

[0, 1]
1.2760867506372389
Layer with highest mlp score: 0
Layer: 0, Number of Heads: 8
Layer: 1, Number of Heads: 5
[5, 6, 7, 8, 9, 10]
3.0063530623623347
Layer with highest mlp score: 5
Layer: 5, Number of Heads: 4
Layer: 6, Number of Heads: 3
Layer: 7, Number of Heads: 2
Layer: 8, Number of Heads: 2
Layer: 9, Number of Heads: 2
Layer: 10, Number of Heads: 2


In [48]:
mha_mlp_contribution

{1: {'n_heads_per_layer': {0: 8, 1: 5}, 'mlp_highest_layer': 0},
 10: {'n_heads_per_layer': {5: 4, 6: 3, 7: 2, 8: 2, 9: 2, 10: 2},
  'mlp_highest_layer': 5}}

In [49]:
list(mha_mlp_contribution.keys())

[1, 10]

In [50]:
for cluster_key in reversed(list(mha_mlp_contribution.keys())):
    cluster       = mha_mlp_contribution[cluster_key]
    layer_indices = sorted(list(cluster['n_heads_per_layer'].keys()))
    mlp_best_idx  = cluster['mlp_highest_layer']
    
    print("Layers in the cluster:", layer_indices)
    print("Best MLP layer:", mlp_best_idx)
    
    layers      = []
    k_per_layer = []
    for layer_idx in layer_indices:
        layers.append(model.bert.encoder.layer[layer_idx])
        n_head = cluster['n_heads_per_layer'][layer_idx]
        k_per_layer.append(n_head)
    
    print("k_per_layer:", k_per_layer)
    
    # --- merge MHA into target (layers[0]) ---
    target_layer_idx = layer_indices[0]
    merged_layer = merge_mha_cluster(model, layers, 0, k_per_layer, device)

    # --- copy best FFN into target if needed ---
    if mlp_best_idx != target_layer_idx:
        print(f"Copying FFN from layer {mlp_best_idx} to target layer {target_layer_idx}")
        target = model.bert.encoder.layer[target_layer_idx]
        source = model.bert.encoder.layer[mlp_best_idx]
        with torch.no_grad():
            target.intermediate.dense.weight.data.copy_(source.intermediate.dense.weight.data.clone())
            target.intermediate.dense.bias.data.copy_(source.intermediate.dense.bias.data.clone())
            target.output.dense.weight.data.copy_(source.output.dense.weight.data.clone())
            target.output.dense.bias.data.copy_(source.output.dense.bias.data.clone())
            target.output.LayerNorm.weight.data.copy_(source.output.LayerNorm.weight.data.clone())
            target.output.LayerNorm.bias.data.copy_(source.output.LayerNorm.bias.data.clone())
    else:
        print(f"Target layer {target_layer_idx} already has best FFN — no copy needed")

    # --- delete source layers from model (reversed so indices stay valid) ---
    for idx in reversed(range(1, len(layer_indices))):
        layer_idx = layer_indices[idx]
        print(f"Deleting layer {layer_idx} from model")
        del model.bert.encoder.layer[layer_idx]

    print(f"Model now has {len(model.bert.encoder.layer)} layers")

Layers in the cluster: [5, 6, 7, 8, 9, 10]
Best MLP layer: 5
k_per_layer: [4, 3, 2, 2, 2, 2]
Merging cluster of 6 layers into target (idx=0)
  k_per_layer (clamped): [4, 3, 2, 2, 2, 2]
  n_head total: 15, new_dim: 960
  Layer (target): taking 4 heads, head_size=64
  Layer: taking 3 heads, head_size=64
  Layer: taking 2 heads, head_size=64
  Layer: taking 2 heads, head_size=64
  Layer: taking 2 heads, head_size=64
  Layer: taking 2 heads, head_size=64
    Wrote 4 heads from layer 0 (target) into slots [0:256]
    Wrote 3 heads from layer 1 into slots [256:448]
    Wrote 2 heads from layer 2 into slots [448:576]
    Wrote 2 heads from layer 3 into slots [576:704]
    Wrote 2 heads from layer 4 into slots [704:832]
    Wrote 2 heads from layer 5 into slots [832:960]
  Done: target now has 15 heads (dim=960)
Target layer 5 already has best FFN — no copy needed
Deleting layer 10 from model
Deleting layer 9 from model
Deleting layer 8 from model
Deleting layer 7 from model
Deleting layer 6 f

In [51]:
model.to(device)

device

device(type='cuda')

# 8. Learning Rate and Leanring Scheduler Setup

In [52]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=3e-5)

In [53]:
from train_eval_func import set_lr_scheduler

num_epochs = 10
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = set_lr_scheduler(
    optimizer          = optimizer,
    num_training_steps = num_training_steps
)

print(num_training_steps)

1800


In [54]:
patience=3

In [55]:
student_layer_ids = [1,2,3,4,5,6]

In [56]:
teacher_layer_ids = []
for layer_idx in layer_composition:
    teacher_layer_ids.append(layer_idx[-1] + 1)

teacher_layer_ids

[2, 3, 4, 5, 11, 12]

In [57]:
layer_composition

[[0, 1], [2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]

In [58]:
import os

In [59]:
save_path = os.path.join(f'./weights/6 layers/{task_name}.pt')

# 8. Training

In [60]:
from train_eval_func import train

In [61]:
train_stats = train(
    model,
    train_dataloader,
    val_dataloader,
    task_name,
    device,
    num_epochs=num_epochs,
    lr=lr_scheduler,
    patience=patience,
    use_early_stopping=True,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    save_path=save_path,
    display_epoch_iter=True,
    teacher_model=teacher_model,
    student_layer_ids=student_layer_ids,
    teacher_layer_ids=student_layer_ids,
    regression=True
)

Training:   0%|          | 0/1800 [00:00<?, ?it/s]

Knowledge distillation used!


/home/abhinavl/work/Layer_Graft/layer-grafting-publication/scratch_train/../train_eval_func.py:240: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)
Training:  10%|▉         | 179/1800 [00:15<02:20, 11.53it/s]

✓ Saved checkpoint (best loss: 0.5368)

<----------------- Epoch 1 ----------------->
Loss: 0.6, Training Metrics:
  pearson: 0.7174
  spearmanr: 0.6771
Validation Loss: 0.54, Validation Metrics:
  pearson: 0.8728
  spearmanr: 0.8700
Elapsed Time: 17.1922 sec


Training:  20%|█▉        | 359/1800 [00:40<02:03, 11.68it/s]

...


<----------------- Epoch 8 ----------------->
Loss: 0.09, Training Metrics:
  pearson: 0.9767
  spearmanr: 0.9737
Validation Loss: 0.53, Validation Metrics:
  pearson: 0.8792
  spearmanr: 0.8752
Elapsed Time: 17.3906 sec


Training:  90%|████████▉ | 1619/1800 [03:12<00:14, 12.31it/s]

✓ Saved checkpoint (best loss: 0.5213)

<----------------- Epoch 9 ----------------->
Loss: 0.08, Training Metrics:
  pearson: 0.9803
  spearmanr: 0.9777
Validation Loss: 0.52, Validation Metrics:
  pearson: 0.8804
  spearmanr: 0.8764
Elapsed Time: 17.4046 sec


Training: 100%|██████████| 1800/1800 [03:46<00:00,  7.96it/s]

✓ Saved checkpoint (best loss: 0.5293)

<----------------- Epoch 10 ----------------->
Loss: 0.08, Training Metrics:
  pearson: 0.9814
  spearmanr: 0.9790
Validation Loss: 0.53, Validation Metrics:
  pearson: 0.8805
  spearmanr: 0.8763
Elapsed Time: 17.3502 sec

Total Training Time: 226.15 sec
Chosen Primary Metric: pearson
Best Metric Run: 0.8805


# 9. Evaluation

In [62]:
best_checkpoint = torch.load(save_path)

In [63]:
model.load_state_dict(best_checkpoint['model_state_dict'])

<All keys matched successfully>

In [65]:
eval_loop(model, val_dataloader, task_name, device, regression=True)

({'pearson': 0.8805024782012627, 'spearmanr': 0.8762564402339595},
 0.5292841937947781)

In [66]:
eval_loop(teacher_model, val_dataloader, task_name, device, regression=True)

({'pearson': 0.900084701022039, 'spearmanr': 0.8962584593578651},
 0.4520208261748578)